# 01 - Exploratory data analysis of findings

Distribution of severity, tier, exposure and the auto-labelled urgency. Uses `data/labelled/labelled_findings.csv` when present, otherwise a synthetic dataset so the notebook always runs.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))   # make the project packages importable from the notebook

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from classifier.feature_engineering import FEATURE_COLUMNS
from classifier.train import make_synthetic_labelled_data
from classifier.labeller import URGENCY_ORDER

labelled = ROOT / "data" / "labelled" / "labelled_findings.csv"
if labelled.exists():
    df = pd.read_csv(labelled, index_col=0)
    source = "real"
else:
    X, y = make_synthetic_labelled_data(1500)
    df = X.assign(urgency=[URGENCY_ORDER[i].value for i in y])
    source = "synthetic"
print(f"{len(df)} findings ({source} data)")
df.head()

## Urgency class balance
Class imbalance motivates SMOTE and macro-F1 as the primary metric.

In [ ]:
order = [u.value for u in URGENCY_ORDER]
counts = df["urgency"].value_counts().reindex(order, fill_value=0)
ax = counts.plot.barh(color="#2a78d6", figsize=(6, 3))
ax.set_xlabel("findings"); ax.set_title("Urgency distribution")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
(counts / counts.sum()).round(3)

## Severity x urgency
How much does context (tier, exposure) move a finding away from its severity default?

In [ ]:
sev_names = {0: "INFORMATIONAL", 1: "LOW", 2: "MEDIUM", 3: "HIGH", 4: "CRITICAL"}
table = pd.crosstab(df["severity_num"].map(sev_names), df["urgency"]).reindex(columns=order, fill_value=0)
table = table.reindex([sev_names[i] for i in range(4, -1, -1) if sev_names[i] in table.index])
table

## Context features

In [ ]:
summary = df.groupby("urgency")[["is_prod", "internet_exposed", "recurrence", "age_days"]].mean().reindex(order).round(2)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df["age_days"].plot.hist(bins=40, ax=axes[0], color="#2a78d6"); axes[0].set_title("Finding age (days)")
df["recurrence"].plot.hist(bins=range(1, 12), ax=axes[1], color="#eb6834"); axes[1].set_title("Recurrence count")
for a in axes: a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()